# sequential-adapt: Colab quickstart

Runs the test suite, a smoke test, and the current grid: **capacity vs
scale** — does paraphrase transfer emerge when coefficient capacity
grows (k, rank), at fixed model scale?

Where this comes from (2026-07-15, corrected coherence re-run —
`docs/CURRENT_STATUS.md` verdicts 18-20): the old "paraphrase 0.0
everywhere" was a probe bug (it measured the frozen base). Measured
honestly, coefficient updates (k=8, r=4) still show ~chance transfer to
the held-out template, BUT full-rank naive_stack LoRA scored .88
consistency in the same runs, and composed states turn out to be
word-keyed, not context-conditioned (off-domain leakage .42-.97).

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

**Colab storage is ephemeral** — run the *Persist artifacts* cell at the
bottom before the VM disconnects, or your results are gone.

In [ ]:
# GPU sanity check
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/sequential_adapt

In [ ]:
# Clone + install (torch is preinstalled on Colab)
import os
if not os.path.isdir("sad-repo"):
    !git clone https://github.com/scottvr/sad-repo
%cd sad-repo
!git pull
!pip install -q transformers

In [ ]:
# HF setup: auth + workarounds. Re-run this cell after any runtime restart —
# env vars do not survive restarts.
import os

# Xet chunked-download backend has known hangs on Colab (downloads stall at
# ~0% forever). Plain HTTP fallback is fast when authenticated.
os.environ["HF_HUB_DISABLE_XET"] = "0"

# Auth from Colab secrets. login() persists the token to
# ~/.cache/huggingface/token, so `!python scripts/...` subprocesses pick it
# up regardless of cell order. Unauthenticated Colab IPs get throttled by
# the Hub, so this matters even for public models like distilgpt2.
# Requires "Notebook access" enabled on the HF secret (key icon, sidebar).
try:
    from google.colab import userdata
    from huggingface_hub import login
    token = userdata.get("HF_TOKEN").strip()
    login(token=token)
    print("HF token set (env + persisted to disk)")
except Exception as e:
    print(f"No HF token ({type(e).__name__}) — public models still work, "
          "but downloads may be rate-limited")

In [ ]:
# Test suite (~10 s, tiny random model; includes the coherence-state
# regression tests — if these fail, nothing below is trustworthy)
!python -m pytest tests -q

In [ ]:
# Smoke test: regression check that the existing pipeline still behaves
# (~1-2 min on GPU; downloads distilgpt2 on first run)
!python scripts/smoke_test.py

In [ ]:
# Multiseed rerun: populates the NEW `final_evals_heldout` column
# (composed-state held-out-template ACCURACY) for every method — this is
# the honest version of the naive_stack-vs-controller comparison, since
# paraphrase *consistency* can be inflated by agreeing-wrong states
# (coeff_add scores .63 while wrong everywhere). 5 seeds to start;
# SEEDS="0 1 2 3 4 5 6 7 8 9" for the full suite.
!SEEDS="0 1 2 3 4" bash scripts/run_multiseed.sh

## Capacity grid (caprank)

Two axes from the k=8/r=4 anchor (96 dims), all replay=1 `--no-gates`:

- components: k = 8 / 16 / 32 / 64 at rank 4 (96 -> 768 dims)
- rank: r = 16 at k = 8 / 16 (component richness)
- combined: k=32 r=16 (k*rank = 512/site, approaching the LoRA regime)

Read the **`heldout acc`** column of the summary: rising with k/rank
means capacity gates paraphrase transfer (cheap fix, TinyLlama can
wait); flat at chance (~.17) while k*rank grows means the constraint is
model scale or the linear-update form itself — Arc 3 is back on.

7 arms x 5 seeds = 35 runs, each ~70 s on a fast card; budget 3-5x on
a T4 (the k=64 and r=16 arms are somewhat slower than baseline).

In [ ]:
# Pilot: 2 seeds (~35 min on a T4)
#!SEEDS="0 1" bash scripts/run_caprank.sh

In [ ]:
# Full grid: 5 seeds — the real thing
!bash scripts/run_caprank.sh

## Render Summary

In [ ]:
# Aggregate and render both summaries inline
!python scripts/summarize_multiseed.py
!python scripts/summarize_sweeps.py
from IPython.display import Markdown, display
display(Markdown(open("artifacts/multiseed_summary.md").read()))
display(Markdown(open("artifacts/sweeps_summary.md").read()))

## Persist artifacts (run before the VM disconnects)

In [ ]:
# Option A: zip + browser download
!zip -qr artifacts.zip artifacts
from google.colab import files
files.download("artifacts.zip")

Reading the table: condition labels are inferred from config fields —
`replay=1;k=32`, `replay=1;rank=16`, `replay=1;k=32;rank=16`, ... Columns
that matter this phase:

- **`heldout acc`** — composed-state accuracy on the never-trained
  template (chance ~.17 for 6 labels). The kill-criterion metric.
- **`paraphrase`** — t0/t2 agreement; only meaningful alongside heldout
  acc and the `coherence_base` floor stored in each JSON.
- `retention` / `cram (newest alone)` — should stay 1.0 / ~floor in every
  arm (replay=1); if not, something regressed.

Full interpretation guide: `docs/CURRENT_STATUS.md` (verdicts 18-20).

In [ ]:
!cp -r artifacts /content/drive/MyDrive/sequential_adapt/